# Image preprocessing

Create a non-destructive, consistently oriented RGB dataset at 224×224 pixels. The stratified source dataset remains unchanged.

In [ ]:
from __future__ import annotations

import hashlib
from collections import Counter
from pathlib import Path

import imagehash
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from PIL import Image, ImageOps, UnidentifiedImageError

SOURCE_ROOT = Path("../datasets/shoe_brands_stratified").resolve()
OUTPUT_ROOT = Path("../datasets/shoe_brands_preprocessed").resolve()
EXPECTED_COUNTS = {"train": 193, "val": 55, "test": 27}
CLASSES = ("adidas", "converse", "nike")
SUPPORTED_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}
OUTPUT_SIZE = (224, 224)
JPEG_QUALITY = 95
PHASH_DISTANCE_THRESHOLD = 4

if not SOURCE_ROOT.is_dir():
    raise FileNotFoundError(
        f"Stratified dataset not found at {SOURCE_ROOT}. Run stratified_split.ipynb first."
    )

## Validate source dataset

In [ ]:
def list_images(directory: Path) -> list[Path]:
    return sorted(
        path for path in directory.glob("*")
        if path.is_file() and path.suffix.lower() in SUPPORTED_EXTENSIONS
    )


source_files = {
    (split_name, class_name): list_images(SOURCE_ROOT / split_name / class_name)
    for split_name in EXPECTED_COUNTS
    for class_name in CLASSES
}
source_counts = {
    split_name: {class_name: len(source_files[(split_name, class_name)]) for class_name in CLASSES}
    for split_name in EXPECTED_COUNTS
}
invalid_counts = [
    f"{split_name}/{class_name}: {image_count}"
    for split_name, class_counts in source_counts.items()
    for class_name, image_count in class_counts.items()
    if image_count != EXPECTED_COUNTS[split_name]
]
if invalid_counts:
    raise ValueError(f"Unexpected source counts: {invalid_counts}")

print(f"Source counts: {source_counts}")
print(f"Total source images: {sum(sum(counts.values()) for counts in source_counts.values())}")

## Apply deterministic preprocessing

In [ ]:
def image_statistics(image: Image.Image) -> tuple[float, float]:
    grayscale_pixels = np.asarray(image.convert("L"), dtype=np.float32)
    return float(grayscale_pixels.mean()), float(grayscale_pixels.std())


preprocessing_records = []
corrupt_files = []
for (split_name, class_name), image_paths in source_files.items():
    output_directory = OUTPUT_ROOT / split_name / class_name
    output_directory.mkdir(parents=True, exist_ok=True)

    for source_path in image_paths:
        output_path = output_directory / f"{source_path.stem}.jpg"
        try:
            with Image.open(source_path) as source_image:
                source_image.load()
                oriented_image = ImageOps.exif_transpose(source_image)
                rgb_image = oriented_image.convert("RGB")
                before_brightness, before_contrast = image_statistics(rgb_image)
                resized_image = rgb_image.resize(OUTPUT_SIZE, Image.Resampling.LANCZOS)
                after_brightness, after_contrast = image_statistics(resized_image)
                resized_image.save(
                    output_path,
                    format="JPEG",
                    quality=JPEG_QUALITY,
                    optimize=True,
                )

                preprocessing_records.append({
                    "split": split_name,
                    "class_name": class_name,
                    "source_path": str(source_path),
                    "output_path": str(output_path),
                    "source_width": source_image.width,
                    "source_height": source_image.height,
                    "source_mode": source_image.mode,
                    "before_brightness": before_brightness,
                    "after_brightness": after_brightness,
                    "before_contrast": before_contrast,
                    "after_contrast": after_contrast,
                    "source_size_kb": source_path.stat().st_size / 1024,
                    "output_size_kb": output_path.stat().st_size / 1024,
                })
        except (OSError, UnidentifiedImageError, ValueError) as error:
            corrupt_files.append({"path": str(source_path), "error": str(error)})

preprocessing_metadata = pd.DataFrame(preprocessing_records)
if corrupt_files:
    raise RuntimeError(f"Preprocessing failed for {len(corrupt_files)} files: {corrupt_files[:3]}")

print(f"Preprocessed images: {len(preprocessing_metadata)}")
print(f"Corrupt images: {len(corrupt_files)}")

## Validate output dataset

In [ ]:
def sha256(file_path: Path) -> str:
    digest = hashlib.sha256()
    with file_path.open("rb") as image_file:
        for chunk in iter(lambda: image_file.read(64 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()


output_records = []
for split_name in EXPECTED_COUNTS:
    for class_name in CLASSES:
        for output_path in list_images(OUTPUT_ROOT / split_name / class_name):
            with Image.open(output_path) as image:
                image.load()
                output_records.append({
                    "split": split_name,
                    "class_name": class_name,
                    "path": str(output_path),
                    "size": image.size,
                    "mode": image.mode,
                    "sha256": sha256(output_path),
                    "phash": imagehash.phash(image),
                })

output_metadata = pd.DataFrame(output_records)
output_counts = (
    output_metadata.groupby(["split", "class_name"]).size().unstack(fill_value=0)
    .reindex(index=EXPECTED_COUNTS, columns=CLASSES, fill_value=0)
)
invalid_output_counts = [
    f"{split_name}/{class_name}: {output_counts.loc[split_name, class_name]}"
    for split_name in EXPECTED_COUNTS
    for class_name in CLASSES
    if output_counts.loc[split_name, class_name] != EXPECTED_COUNTS[split_name]
]
if invalid_output_counts:
    raise RuntimeError(f"Unexpected output counts: {invalid_output_counts}")
if set(output_metadata["size"]) != {OUTPUT_SIZE}:
    raise RuntimeError(f"Unexpected output dimensions: {set(output_metadata['size'])}")
if set(output_metadata["mode"]) != {"RGB"}:
    raise RuntimeError(f"Unexpected output modes: {set(output_metadata['mode'])}")

cross_split_exact_duplicates = (
    output_metadata.groupby("sha256")["split"].nunique().loc[lambda counts: counts > 1]
)
if not cross_split_exact_duplicates.empty:
    raise RuntimeError(
        f"Found {len(cross_split_exact_duplicates)} exact duplicate groups across splits."
    )

display(output_counts)
print(f"Output images: {len(output_metadata)}")
print(f"Output dimensions: {set(output_metadata['size'])}")
print(f"Output modes: {set(output_metadata['mode'])}")
print("Cross-split exact duplicates: 0")

## Perceptual near-duplicate report

In [ ]:
near_duplicate_pairs = []
records = output_metadata.to_dict("records")
for left_index, left_record in enumerate(records):
    for right_record in records[left_index + 1:]:
        if left_record["split"] == right_record["split"]:
            continue
        phash_distance = left_record["phash"] - right_record["phash"]
        if phash_distance <= PHASH_DISTANCE_THRESHOLD:
            near_duplicate_pairs.append({
                "left_split": left_record["split"],
                "left_class": left_record["class_name"],
                "left_path": left_record["path"],
                "right_split": right_record["split"],
                "right_class": right_record["class_name"],
                "right_path": right_record["path"],
                "phash_distance": phash_distance,
            })

near_duplicates = pd.DataFrame(near_duplicate_pairs)
print(f"Cross-split near-duplicate candidates (pHash distance <= {PHASH_DISTANCE_THRESHOLD}): {len(near_duplicates)}")
if not near_duplicates.empty:
    display(near_duplicates.head(20))
    print("Review these candidates manually; no images were removed automatically.")

## Before-and-after quality comparison

In [ ]:
quality_comparison = preprocessing_metadata[[
    "before_brightness", "after_brightness",
    "before_contrast", "after_contrast",
    "source_size_kb", "output_size_kb",
]].describe().round(2)
display(quality_comparison)

sample_rows = preprocessing_metadata.sample(n=6, random_state=37)
figure, axes = plt.subplots(len(sample_rows), 2, figsize=(8, 3 * len(sample_rows)))
for row_index, (_, row) in enumerate(sample_rows.iterrows()):
    with Image.open(row["source_path"]) as source_image:
        axes[row_index, 0].imshow(source_image)
    with Image.open(row["output_path"]) as output_image:
        axes[row_index, 1].imshow(output_image)
    axes[row_index, 0].set_title(f"Before: {row['split']} / {row['class_name']}")
    axes[row_index, 1].set_title("After: RGB 224x224")
    axes[row_index, 0].axis("off")
    axes[row_index, 1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
print("PREPROCESSING SUMMARY")
print(f"- Source images: {len(preprocessing_metadata)}")
print(f"- Output images: {len(output_metadata)}")
print("- Output format: RGB JPEG")
print(f"- Output resolution: {OUTPUT_SIZE[0]}x{OUTPUT_SIZE[1]}")
print("- Cross-split exact duplicates: 0")
print(f"- Near-duplicate candidates for manual review: {len(near_duplicates)}")
print("- Static augmentation was not generated. Apply augmentation dynamically to train only.")
print("- Training was not started.")